# generate_retailmax_data.ipynb

Genera los datos sintéticos del sistema transaccional de **RetailMax**
(Escenario B — Retail y Comercio Electrónico), cubriendo las entidades
necesarias para el modelamiento dimensional y los procesos de ML/análisis
posteriores. Produce las siguientes tablas exigidas por la prueba técnica:

- `MSTR_PROVEEDORES`
- `MSTR_ARTICULOS`
- `MSTR_TIENDAS`
- `CRM_MIEMBROS`
- `TRANS_VENTAS`
- `INV_STOCK_DIARIO`
- `POST_DEVOLUCIONES`

## Características

- **Reproducible** mediante semilla aleatoria fija (`config.yaml`)
- **Distribuciones realistas**: horarios pico, edades normales, Pareto en ventas
- **Integridad referencial** entre tablas de hechos y dimensiones
- **~5% de nulos controlados** en campos no críticos
- **Cobertura temporal** ≥ 12 meses
- **Anomalías intencionales documentadas** (ver [`docs/anomalias.md`](docs/anomalias.md))
- **Salida en múltiples formatos**: CSV, JSON

## Uso

```bash
python generate_data.py --config config.yaml

# Para pruebas rápidas (muestra reducida)
python generate_data.py --config config.yaml --sample 0.01
```

## Instalar las librerías

Este proyecto utiliza las siguientes librerías para la generación de datos sintéticos de RetailMax:

- **Faker**: genera datos sintéticos realistas (nombres, direcciones, correos, fechas, etc.) en las tablas dimensionales como `MSTR_ARTICULOS`, `CRM_MIEMBROS` y `MSTR_TIENDAS`.
- **PyYAML**: lee el archivo de configuración `config.yaml` (semilla, volúmenes, rango de fechas, formatos de salida).
- **pandas**: estructura y exporta los datos generados a los distintos formatos (CSV, JSON).
- **numpy**: genera las distribuciones numéricas (edades normales, precios lognormal, horarios pico, etc.) de forma vectorizada y reproducible.

In [1]:
%pip install faker
%pip install PyYAML
%pip install pandas
%pip install numpy
%pip install scipy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### Importar las librerías

Se importan las librerías necesarias para la generación de datos sintéticos: `Faker` para datos realistas, `yaml` para leer la configuración, `pandas`/`numpy` para estructurar y generar las distribuciones de datos, y las utilidades estándar (`argparse`, `json`, `os`, `random`, `time`, `datetime`) para el manejo de parámetros, formatos de salida, semillas de reproducibilidad y fechas.

In [2]:
from faker import Faker
import yaml
import pandas as pd
import numpy as np

import argparse
import json
import os
import random
import time
from datetime import datetime, timedelta
from scipy.stats import truncnorm

### Cargar configuración y fijar semilla

In [3]:
# Abrimos el archivo de configuración y cargamos los datos
with open("config.yaml", "r", encoding="utf-8") as archivo:
    datos = yaml.full_load(archivo)

# Extraemos la semilla del archivo de configuración
semilla = datos["semilla"]

# Fijamos la semilla para que siempre devuelva los mismos datos
Faker.seed(semilla)
random.seed(semilla)
np.random.seed(semilla)

fake = Faker("es_CO")  # Configuramos Faker para generar datos en español (Colombia)

### Extracción de parámetros de configuración

Se cargan en variables individuales todos los parámetros definidos en el archivo de configuración (`datos`), organizados por tipo:

- **Distribuciones**: tipos de tienda, canales, macro categorías, rangos de precios por categoría, países, centros de distribución, stock, franjas horarias, días de la semana, edades, bins de rango de edad, género, tipos de pago, descuento aplicado, motivos y estados de devolución, calificación de proveedores, activos y anomalías.
- **Fechas**: rango de fechas a generar.
- **Volúmenes**: cantidad de registros a simular por entidad.
- **Otros**: porcentaje de valores nulos a inyectar y formatos de salida del dataset.

Estas variables se utilizarán posteriormente como parámetros de entrada para las funciones de generación de datos sintéticos.

In [4]:
# Distribuciones
tipos_tienda = datos["tipos_tienda"]
canales = datos["canales"]
macro_categorias = datos["macro_categorias"]
rango_precios_por_categoria = datos["rango_precios_por_categoria"]
paises = datos["paises"]
centros_distribucion = datos["centros_distribucion"]
stock = datos["stock"]
franjas_horarias = datos["franjas_horarias"]
dias_semana = datos["dias_semana"]
edades = datos["edades"]
rango_edad_bins = datos["rango_edad_bins"]
genero = datos["genero"]
tipos_pago = datos["tipos_pago"]
descuento_aplicado = datos["descuento_aplicado"]
motivos_devolucion = datos["motivos_devolucion"]
estados_devolucion = datos["estados_devolucion"]
calificacion_proveedores = datos["calificacion_proveedores"]
activos = datos["activos"]
anomalias = datos["anomalias"]

# Fechas
fecha_inicio = datos["rango_fechas"]["inicio"]
fecha_fin = datos["rango_fechas"]["fin"]

# Volúmenes
volumenes = datos["volumenes"]

porcentaje_nulos = datos["valores_nulos"]
formatos_salida = datos["formatos_salida"]

### Conversión de fechas a formato `datetime`

Las variables `fecha_inicio` y `fecha_fin`, extraídas de la configuración como cadenas de texto (`str`) en formato `"YYYY-MM-DD"`, se convierten a objetos `datetime` mediante `datetime.strptime`.

Esta conversión es necesaria para poder realizar operaciones de fecha (comparaciones, cálculo de rangos, generación de fechas aleatorias, etc.) en los pasos posteriores del proceso de simulación de datos.

In [5]:
fecha_inicio = datetime.strptime(fecha_inicio, "%Y-%m-%d")
fecha_fin = datetime.strptime(fecha_fin, "%Y-%m-%d")

## Funciones auxiliares para la generación de datos

Con el fin de mantener un código modular, reutilizable y fácil de mantener, se implementó un conjunto de funciones auxiliares encargadas de generar los distintos atributos utilizados en las tablas del proyecto. Cada función obtiene la información necesaria desde el archivo de configuración (`config.yaml`) y genera valores sintéticos respetando las distribuciones y restricciones definidas para RetailMax.

---

### Generación de fecha y hora de la transacción

Para simular el momento exacto en que ocurre cada venta, se implementaron tres funciones auxiliares que generan la fecha, seleccionan una franja horaria y calculan una hora específica dentro de esa franja. Este enfoque permite obtener distribuciones temporales coherentes con el comportamiento esperado de un entorno de retail, en lugar de una distribución uniforme durante todo el día.

#### `generar_fecha(fecha_inicio, fecha_fin)`

Genera una fecha aleatoria dentro del rango definido en el archivo de configuración utilizando `fake.date_between()`. Esto garantiza que todas las transacciones se encuentren dentro del período histórico establecido para la simulación.

#### `generar_franja(franjas_horarias)`

Selecciona una franja horaria mediante una distribución ponderada, respetando las probabilidades configuradas para cada intervalo del día.

#### `generar_hora_en_franja(franja)`

Genera una hora aleatoria dentro del intervalo correspondiente a la franja seleccionada:

1. Convierte las horas de inicio y fin a objetos `datetime`.
2. Calcula el intervalo disponible mediante `timedelta`.
3. Genera un desplazamiento aleatorio en segundos.
4. Obtiene la hora final sumando dicho desplazamiento a la hora de inicio.

#### Flujo de generación

```text
fecha_inicio, fecha_fin
        │
        ▼
  generar_fecha()
        │
        ▼
 generar_franja()
        │
        ▼
generar_hora_en_franja()
        │
        ▼
Fecha y hora de la transacción
```

---

### Generación de información demográfica

Las siguientes funciones permiten generar la información básica de los miembros del programa de fidelización.

#### `generar_edad(edades)`

Genera una edad utilizando una distribución normal truncada, respetando la media, la desviación estándar y los límites mínimo y máximo definidos en la configuración.

#### `generar_rango_edades(edad, rango_edad_bins)`

Clasifica la edad generada dentro del rango correspondiente según la configuración (`18-25`, `26-35`, `36-45`, etc.).

#### `generar_genero(genero)`

Selecciona un género mediante una distribución ponderada utilizando las probabilidades definidas en el archivo de configuración.

---

### Generación de ubicación geográfica

Estas funciones permiten asignar un país y una ciudad coherentes para cada registro.

#### `generar_pais(paises)`

Selecciona un país utilizando las probabilidades definidas para cada uno.

#### `generar_ciudad(pais, paises)`

Una vez seleccionado el país, genera aleatoriamente una ciudad perteneciente a dicho país.

---

### Generación de productos

Las siguientes funciones generan la información relacionada con los artículos comercializados.

#### `generar_categoria(macro_categorias)`

Selecciona una categoría de producto respetando la distribución configurada.

#### `generar_precio(categoria, rango_precios_por_categoria)`

Genera un precio aleatorio dentro del rango permitido para la categoría seleccionada.

---

### Generación de tiendas y canales

#### `generar_tipo_tienda(tipos_tienda)`

Selecciona el tipo de tienda mediante una distribución ponderada.

#### `generar_canal(canales)`

Genera el canal por el cual se realiza la venta (tienda física, e-commerce o marketplace), respetando las probabilidades definidas.

---

### Generación de información comercial

#### `generar_tipo_pago(tipos_pago)`

Selecciona el método de pago de acuerdo con la distribución configurada.

#### `generar_descuento_aplicado(descuento_aplicado)`

Determina si una venta recibe descuento. En caso afirmativo, genera un porcentaje aleatorio dentro del rango establecido; de lo contrario, retorna un descuento de `0.0`.

---

### Generación de proveedores

#### `generar_calificacion_proveedores(calificacion_proveedores)`

Genera una calificación utilizando una distribución normal truncada, garantizando valores comprendidos entre el mínimo y el máximo configurados.

---

### Generación del estado de registros

#### `generar_activo(activos, tipo)`

Determina si un registro se encuentra activo o inactivo, utilizando la probabilidad correspondiente al tipo de entidad (artículos, proveedores, tiendas o miembros).

---

### Generación de devoluciones

#### `generar_estado_devolucion(estados_devolucion)`

Selecciona aleatoriamente el estado de una devolución según la distribución configurada.

#### `generar_motivo_devolucion(motivos_devolucion)`

Selecciona un motivo de devolución respetando las probabilidades definidas y devuelve toda la información asociada al motivo.

---

### Generación de centros de distribución

#### `generar_centro_distribucion(pais, centros_distribucion)`

Obtiene un centro de distribución compatible con el país previamente generado, garantizando coherencia geográfica en los datos.

---

### Generación de inventario

Para simular el comportamiento del inventario se implementaron tres funciones complementarias.

#### `generar_stock_minimo(stock)`

Genera el nivel mínimo de inventario permitido dentro del rango configurado.

#### `generar_stock_maximo(stock, stock_minimo)`

Genera el nivel máximo de inventario asegurando que siempre sea mayor o igual al stock mínimo.

#### `generar_stock_fisico(stock, stock_minimo, stock_maximo)`

Calcula el stock físico aplicando una variación porcentual sobre los niveles mínimo y máximo definidos en la configuración, obteniendo un valor coherente para la simulación diaria.

---

En conjunto, estas funciones constituyen la base del generador de datos sintéticos del proyecto, permitiendo construir registros consistentes y reproducibles a partir de las reglas de negocio establecidas en el archivo `config.yaml`.

In [6]:
def generar_fecha(fecha_inicio, fecha_fin):
    fecha_aleatoria = fake.date_between(start_date=fecha_inicio, end_date=fecha_fin)
    return fecha_aleatoria


def generar_franja(franjas_horarias):
    probabilidades = [franja["probabilidad"] for franja in franjas_horarias]
    franja_aleatoria = np.random.choice(franjas_horarias, p=probabilidades)
    return franja_aleatoria


def generar_hora_en_franja(franja):
    inicio = datetime.strptime(franja["inicio"], "%H:%M")
    fin = datetime.strptime(franja["fin"], "%H:%M")
    delta = fin - inicio
    segundos_aleatorios = random.randint(0, int(delta.total_seconds()))
    hora_aleatoria = inicio + timedelta(seconds=segundos_aleatorios)
    return hora_aleatoria.time()


def generar_edad(edades):
    # Definir límites de edad deseados y parámetros
    media = edades["media"]
    desviacion = edades["desviacion"]
    edad_min = edades["minimo"]
    edad_max = edades["maximo"]
    # Estandarizar los límites
    a, b = (edad_min - media) / desviacion, (edad_max - media) / desviacion
    # Generar datos truncados
    edad_trunc = truncnorm.rvs(a, b, loc=media, scale=desviacion)
    edad_final = int(np.round(edad_trunc))
    return edad_final


def generar_rango_edades(edad, rango_edad_bins):
    for rango in rango_edad_bins:
        if rango["min"] <= edad <= rango["max"]:
            return rango["rango"]
    return None


def generar_genero(genero):
    nombres = [g["nombre"] for g in genero]
    probabilidades = [g["probabilidad"] for g in genero]
    genero_aleatorio = np.random.choice(nombres, p=probabilidades)
    return genero_aleatorio


def generar_pais(paises):
    nombres = list(paises.keys())
    probabilidades = [p["probabilidad"] for p in paises.values()]

    pais_aleatorio = np.random.choice(nombres, p=probabilidades)
    return pais_aleatorio


def generar_ciudad(pais, paises):
    ciudades = paises[pais]["ciudades"]
    ciudad_aleatoria = np.random.choice(ciudades)
    return ciudad_aleatoria


def generar_categoria(macro_categorias):
    categorias = [cat["nombre"] for cat in macro_categorias]
    probabilidades = [cat["probabilidad"] for cat in macro_categorias]
    categoria_aleatoria = np.random.choice(categorias, p=probabilidades)
    return categoria_aleatoria

def generar_tipo_tienda(tipos_tienda):
    nombres = [t["nombre"] for t in tipos_tienda]
    probabilidades = [t["probabilidad"] for t in tipos_tienda]
    tipo_tienda_aleatorio = np.random.choice(nombres, p=probabilidades)
    return tipo_tienda_aleatorio

def generar_canal(canales):
    nombres = [c["nombre"] for c in canales]
    probabilidades = [c["probabilidad"] for c in canales]
    canal_aleatorio = np.random.choice(nombres, p=probabilidades)
    return canal_aleatorio

def generar_tipo_pago(tipos_pago):
    nombres = [t["nombre"] for t in tipos_pago]
    probabilidades = [t["probabilidad"] for t in tipos_pago]
    tipo_pago_aleatorio = np.random.choice(nombres, p=probabilidades)
    return tipo_pago_aleatorio

def generar_descuento_aplicado(descuento_aplicado):
    probabilidad = descuento_aplicado['probabilidad_con_descuento']
    if np.random.random() < probabilidad:
        minimo = descuento_aplicado["rango_porcentaje"]["min"]
        maximo = descuento_aplicado["rango_porcentaje"]["max"]
        return round(np.random.uniform(minimo, maximo), 2)
    return 0.0

def generar_precio(categoria, rango_precios_por_categoria):
    precio_min = rango_precios_por_categoria[categoria]["min"]
    precio_max = rango_precios_por_categoria[categoria]["max"]
    precio = np.random.uniform(precio_min, precio_max)
    return round(precio, 2)

def generar_calificacion_proveedores(calificacion_proveedores):
    media = calificacion_proveedores["media"]
    desviacion = calificacion_proveedores["desviacion"]
    calificacion_min = calificacion_proveedores["minimo"]
    calificacion_max = calificacion_proveedores["maximo"]

    a = (calificacion_min - media) / desviacion
    b = (calificacion_max - media) / desviacion

    calificacion_trunc = truncnorm.rvs(
        a, b,
        loc=media,
        scale=desviacion
    )

    calificacion_final = int(np.round(calificacion_trunc))
    return calificacion_final

def generar_activo(activos, tipo):
    probabilidad = activos[tipo]
    return np.random.random() < probabilidad

def generar_estado_devolucion(estados_devolucion):
    nombre = [dev["nombre"] for dev in estados_devolucion]
    probabilidades = [dev["probabilidad"] for dev in estados_devolucion]
    estados_devolucion_aleatorio = np.random.choice(nombre, p=probabilidades)
    return estados_devolucion_aleatorio

def generar_motivo_devolucion(motivos_devolucion):
    probabilidades = [m["probabilidad"] for m in motivos_devolucion]
    motivo = np.random.choice(motivos_devolucion, p=probabilidades)
    return motivo

def generar_centro_distribucion(pais, centros_distribucion):
    centros = [
        centro for centro in centros_distribucion if centro["pais_asociado"] == pais
    ]

    return np.random.choice(centros)

def generar_stock_minimo(stock):
    minimo = stock["stock_minimo_config"]["min"]
    maximo = stock["stock_minimo_config"]["max"]

    return random.randint(minimo, maximo)

def generar_stock_maximo(stock, stock_minimo):
    maximo_config = stock["stock_maximo_config"]["max"]

    return random.randint(stock_minimo, maximo_config)

def generar_stock_fisico(stock, stock_minimo, stock_maximo):
    variacion = stock["stock_fisico_variacion"]

    limite_inferior = int(stock_minimo * (1 - variacion))
    limite_superior = int(stock_maximo * (1 + variacion))

    stock_fisico = random.randint(
        max(0, limite_inferior),
        limite_superior
    )

    return stock_fisico

fecha = generar_fecha(fecha_inicio, fecha_fin)
print(fecha)

franja = generar_franja(franjas_horarias)
print(franja)

hora = generar_hora_en_franja(franja)
print(hora)

edad = generar_edad(edades)
print(edad)

rango_edad = generar_rango_edades(edad, rango_edad_bins)
print(rango_edad)

genero_salida = generar_genero(genero)
print(genero_salida)

pais_salida = generar_pais(paises)
print(pais_salida)

ciudad_salida = generar_ciudad(pais_salida, paises)
print(ciudad_salida)

categoria = generar_categoria(macro_categorias)
print(categoria)

tienda = generar_tipo_tienda(tipos_tienda)
print(tienda)

canal = generar_canal(canales)
print(canal)

tipo_pago = generar_tipo_pago(tipos_pago)
print(tipo_pago)

descuento = generar_descuento_aplicado(descuento_aplicado)
print(descuento)

precio = generar_precio(categoria, rango_precios_por_categoria)
print(precio)

calificacion = generar_calificacion_proveedores(calificacion_proveedores)
print(calificacion)

activo_articulo = generar_activo(activos, "articulos")
print(activo_articulo)

activo_proveedor = generar_activo(activos, "proveedores")
print(activo_proveedor)

activo_tienda = generar_activo(activos, "tiendas")
print(activo_tienda)

activo_miembro = generar_activo(activos, "miembros")
print(activo_miembro)

estados = generar_estado_devolucion(estados_devolucion)
print(estados)

motivo = generar_motivo_devolucion(motivos_devolucion)
print(motivo)

pais = generar_pais(paises)
centro = generar_centro_distribucion(pais, centros_distribucion)
print(centro)

stock_minimo = generar_stock_minimo(stock)
stock_maximo = generar_stock_maximo(stock, stock_minimo)
stock_fisico = generar_stock_fisico(stock, stock_minimo, stock_maximo)

print(stock_minimo)
print(stock_maximo)
print(stock_fisico)

2025-09-20
{'nombre': 'mediodia', 'inicio': '11:00', 'fin': '14:00', 'probabilidad': 0.3}
13:54:36
58
46-60
femenino
mexico
Monterrey
Cuidado personal e higiene
hipermercado
tienda
tarjeta_credito
0.21
9230.87
4
False
True
False
True
aprobada
{'codigo': 'DEF', 'descripcion': 'Producto defectuoso', 'probabilidad': 0.3}
{'nombre': 'CD_Bogota', 'pais_asociado': 'colombia', 'tiempo_repo_dias': {'min': 1, 'max': 3}}
24
49
63


Guardar todas las lebrerias con las versiones instaladas funcionales para el proyecto

In [7]:
pip freeze > requirements.txt

Note: you may need to restart the kernel to use updated packages.
